<a href="https://colab.research.google.com/github/shribr/Bricky/blob/main/Tools/dinov2-embeddings/Bricky-evaluate-dinov2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DINOv2 ViT-L/14 Evaluation Run

Embeds the full catalog with DINOv2 ViT-L/14, scores retrieval, and saves the report to Google Drive.

**Runtime:** Select **Runtime → Change runtime type → GPU** (T4 is sufficient).

## 1. Clone repo & install dependencies

In [ ]:
import os

REPO = "/content/Bricky"
if not os.path.exists(REPO):
    !git clone https://github.com/shribr/Bricky.git /content/Bricky
else:
    print("Bricky/ already exists — skipping clone.")

%cd /content/Bricky
!pip install -q -r Tools/dinov2-embeddings/requirements.txt

## 2. Mount Google Drive & create output directories

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

# Create output directories on Drive and locally
for d in [
    "/content/drive/MyDrive/Bricky/reports",
    "Tools/dinov2-embeddings/index",
    "Tools/dinov2-embeddings/reports",
    "Tools/dinov2-embeddings/eval",
]:
    os.makedirs(d, exist_ok=True)
    print(f"  ✓ {d}")
print("Output directories ready.")

## 3. Build the evaluation set

Deterministic given the seed. Skips if already present.

In [ ]:
if not os.path.exists("Tools/dinov2-embeddings/eval/ground_truth.json"):
    !python Tools/dinov2-embeddings/build_eval_set.py --count 200 --variants-per-figure 8 --seed 42
else:
    print("Eval set already exists — skipping.")

## 4. Apply bbox-crop patch

Adds a `detect_figure_bbox` function to `evaluate_retrieval.py` so queries are auto-cropped before embedding. Idempotent — skips if already patched.

In [ ]:
from pathlib import Path

p = Path("Tools/dinov2-embeddings/evaluate_retrieval.py")
txt = p.read_text()
if "detect_figure_bbox" not in txt:
    FN = '''

def detect_figure_bbox(img):
    import numpy as np
    arr = np.asarray(img.convert("RGB"), dtype=np.float32)
    corners = np.concatenate([arr[0:5].reshape(-1,3), arr[-5:].reshape(-1,3),
                              arr[:,0:5].reshape(-1,3), arr[:,-5:].reshape(-1,3)])
    bg = corners.mean(axis=0)
    dist = np.sqrt(((arr - bg) ** 2).sum(axis=2))
    ys, xs = np.where(dist > 40)
    if len(xs) == 0:
        return (0, 0, img.width, img.height)
    pad = 4
    return (max(0,int(xs.min())-pad), max(0,int(ys.min())-pad),
            min(img.width,int(xs.max())+pad), min(img.height,int(ys.max())+pad))
'''
    txt = txt.replace("K_RECALL = [1, 5, 10, 50]", "K_RECALL = [1, 5, 10, 50]" + FN, 1)
    txt = txt.replace(
        '        img = Image.open(p).convert("RGB")\n        crop = torso_crop(img)',
        '        img = Image.open(p).convert("RGB")\n'
        '        img = img.crop(detect_figure_bbox(img))\n'
        '        crop = torso_crop(img)', 1)
    p.write_text(txt)
    print("Evaluator patched with bbox-crop.")
else:
    print("Patch already applied — skipping.")

## 5. Embed catalog with ViT-L/14

In [ ]:
!python Tools/dinov2-embeddings/embed_catalog.py \
    --model dinov2_vitl14 \
    --out Tools/dinov2-embeddings/index/dinov2_vitl14

## 6. Evaluate retrieval & save report to Drive

In [ ]:
REPORT = "/content/drive/MyDrive/Bricky/reports/dinov2_vitl14.json"

!python Tools/dinov2-embeddings/evaluate_retrieval.py \
    --index Tools/dinov2-embeddings/index/dinov2_vitl14 \
    --report {REPORT}

# Dump the report inline
print("\n\n=============== REPORT ({}) ===============".format(REPORT))
!cat {REPORT}